# 02. Resources 与 Prompts

除了 Tool，MCP 还定义了两种原语来为 Agent 提供「原料」和「模板」：

- **Resource**：Agent 可以读取的数据（文件、数据库记录、API 响应）
- **Prompt**：预定义的提示词模板，支持参数化

> **检查点**：能区分 Tool（让 Agent 做事）、Resource（给 Agent 看数据）、Prompt（帮 Agent 组织语言）三种原语的用途。

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'xmake.lua').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'agents'))

from scripts.nb_helpers import setup, chat, chat_json, chat_with_tools, show_trace
client = setup()


🌐 API 模式 — model=deepseek-v4-flash  base_url=https://api.deepseek.com


In [2]:
# Resources: 暴露文件系统内容
from dataclasses import dataclass

@dataclass
class Resource:
    uri: str; name: str; mimeType: str = "text/plain"
    def read(self) -> str: pass

class FileResource(Resource):
    def __init__(self, uri: str, path: str):
        super().__init__(uri, f"File: {path}")
        self.path = path
    def read(self) -> str:
        try:
            with open(self.path) as f:
                return f.read()
        except FileNotFoundError:
            return f"Error: file not found: {self.path}"

# Prompts: 参数化提示词模板
class PromptTemplate:
    def __init__(self, name: str, description: str, template: str, arguments: list[dict]):
        self.name = name; self.description = description
        self.template = template; self.arguments = arguments
    def render(self, **kwargs) -> str:
        result = self.template
        for k, v in kwargs.items():
            result = result.replace(f"{{{{{k}}}}}", str(v))
        return result

# 注册 resources 和 prompts
resources = [
    FileResource("file:///config.json", "/tmp/config.json"),
]
prompts = [
    PromptTemplate("code-review", "Review code for bugs",
        "Review this {{language}} code for bugs and style issues:\n```\n{{code}}\n```",
        [{"name":"language","required":True},{"name":"code","required":True}]),
    PromptTemplate("summarize", "Summarize text",
        "Summarize the following in {{style}} style:\n{{text}}",
        [{"name":"text","required":True},{"name":"style","required":False}]),
]

print("Resources:")
for r in resources:
    print(f"  {r.uri} ({r.name})")

print("\nPrompt templates:")
for p in prompts:
    args_desc = ", ".join(f"{a['name']}{'*' if a.get('required') else ''}" for a in p.arguments)
    print(f"  {p.name}: {p.description} (args: {args_desc})")

print("\nRendered prompt:")
print(prompts[0].render(language="Python", code="def add(a,b): return a-b"))

Resources:
  file:///config.json (File: /tmp/config.json)

Prompt templates:
  code-review: Review code for bugs (args: language*, code*)
  summarize: Summarize text (args: text*, style)

Rendered prompt:
Review this Python code for bugs and style issues:
```
def add(a,b): return a-b
```


## Resource 安全

Resource 暴露了数据，也暴露了攻击面：
- **路径穿越**：Agent 传入 `../../etc/passwd` 读取敏感文件
- **SSRF**：Resource URI 指向内网地址
- **数据泄露**：Resource 返回的数据包含 PII 但未脱敏

防御：URI 白名单、路径规范化校验、输出过滤。

> **检查点**：能为 Resource 设计安全校验层（输入 URI 校验 + 输出内容过滤）。

## 真实 API 实验

MCP prompts/get 的价值：服务端统一管理的提示词模板。填充模板后发给真实 LLM，体会「prompt 即资源」。


In [3]:
# 模拟 prompts/get 返回的模板 → 填充 → 真实调用
prompt_template = {"name": "code-review", "arguments": {"language": "Python", "code": "def add(a,b): return a+b"}}
filled = f"请审查以下 {prompt_template['arguments']['language']} 代码，指出 2 个改进点：\n\n{prompt_template['arguments']['code']}"

review = chat(client, filled, system="你是严格的 code reviewer，输出编号列表。", max_tokens=200)
print(review)
print("\n观察：模板由「服务端」定义，client 只填参数 — 团队可以统一迭代 prompt 而不改客户端代码。")




观察：模板由「服务端」定义，client 只填参数 — 团队可以统一迭代 prompt 而不改客户端代码。


## 练习

1. 实现 `DatabaseResource`：从 SQLite 数据库读取记录，URI 格式为 `db://table/row_id`。
2. 实现 `prompts/list` 和 `prompts/get` 端点（JSON-RPC 格式），支持参数渲染。
3. 设计一个「A/B 测试」的 Prompt 管理系统：同一个场景有多套 prompt，自动选择效果最好的。

> **检查点**：能为你的业务场景设计至少 2 个 Resource 和 2 个 Prompt Template。